In [144]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix

In [145]:
df = pd.read_csv("matches.csv")

home = df[df["venue"].str.lower() == "home"].copy()
away = df[df["venue"].str.lower() == "away"].copy()

matches = pd.merge(
    home,
    away,
    left_on=["date", "comp", "opponent"],
    right_on=["date", "comp", "team"],
    suffixes=("_home", "_away"),
    how="inner"
)

matches = pd.DataFrame({
    "date": pd.to_datetime(matches["date"]),
    "home_team": matches["team_home"],
    "away_team": matches["team_away"],
    "home_goals": pd.to_numeric(matches["gf_home"], errors="coerce"),
    "away_goals": pd.to_numeric(matches["gf_away"], errors="coerce")
})

def get_result(r):
    if r.home_goals > r.away_goals: return "H"
    if r.home_goals < r.away_goals: return "A"
    return "D"

matches["result"] = matches.apply(get_result, axis=1)


In [146]:
# --- Elo Ratings ---
K = 20
START_ELO = 1500
elos = {}
home_elos, away_elos = [], []

for _, row in matches.sort_values("date").iterrows():
    h, a = row["home_team"], row["away_team"]
    elos.setdefault(h, START_ELO)
    elos.setdefault(a, START_ELO)

    rh, ra = elos[h], elos[a]
    home_elos.append(rh)
    away_elos.append(ra)

    # Compute match score
    if row.home_goals > row.away_goals: score = 1
    elif row.home_goals < row.away_goals: score = 0
    else: score = 0.5

    # Home advantage
    adv = 100
    expected_h = 1 / (1 + 10 ** ((ra - (rh + adv)) / 400))
    expected_a = 1 - expected_h

    # Update Elo ratings
    elos[h] = rh + K * (score - expected_h)
    elos[a] = ra + K * ((1 - score) - expected_a)

matches["home_elo"] = home_elos
matches["away_elo"] = away_elos
matches["elo_diff"] = matches["home_elo"] - matches["away_elo"]

In [147]:
# --- Model Training ---
features = ["elo_diff", "home_elo", "away_elo"]
X = matches[features]
y = matches["result"]

le = LabelEncoder()
y_enc = le.fit_transform(y)

split = int(len(matches) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y_enc[:split], y_enc[split:]

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)
clf.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [148]:
# Predictions & Evaluation
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Log loss:", log_loss(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.29069767441860467
Log loss: 1.3101960567298718

Classification Report:
               precision    recall  f1-score   support

           A       0.60      0.31      0.41        49
           D       0.18      0.15      0.16        20
           H       0.16      0.41      0.23        17

    accuracy                           0.29        86
   macro avg       0.31      0.29      0.27        86
weighted avg       0.41      0.29      0.31        86

Confusion Matrix:
 [[15  8 26]
 [ 6  3 11]
 [ 4  6  7]]


In [149]:
def predict_match(home_team, away_team, venue="home"):
    if home_team not in elos or away_team not in elos:
        return "One of the teams is not in the Elo database."
    
    home_elo = elos[home_team]
    away_elo = elos[away_team]

    if venue.lower() == "away":
        home_elo, away_elo = away_elo, home_elo

    elo_diff = home_elo - away_elo

    X_new = pd.DataFrame([{
        "elo_diff": elo_diff,
        "home_elo": home_elo,
        "away_elo": away_elo
    }])

    pred_class = clf.predict(X_new)[0]
    pred_proba = clf.predict_proba(X_new)[0]

    result_label = le.inverse_transform([pred_class])[0]
    prob_dict = dict(zip(le.classes_, pred_proba))

    # Friendly text output
    friendly_text = (
        f"Predicted outcome: {result_label}\n"
        f"{home_team} win probability: {prob_dict['H']*100:.1f}%\n"
        f"Draw probability: {prob_dict['D']*100:.1f}%\n"
        f"{away_team} win probability: {prob_dict['A']*100:.1f}%"
    )

    return friendly_text


In [150]:
print(predict_match("Arsenal", "Chelsea", venue="home"))

Predicted outcome: A
Arsenal win probability: 14.4%
Draw probability: 31.4%
Chelsea win probability: 54.2%
